# ESPIRiT: is a 20x20 ACS enough?

The brain maps on disk were made by `scripts/make_espirit_smaps.py` with
`--acs 20`, and the recon nets that consume them underperform an E2E-VarNet,
which estimates its own maps. This notebook estimates maps at the shipped ACS
and at a large one on the same slice and compares the **coil-combined images**
`x = sum_c conj(s_c) c_c` -- the ground truth every net is trained to produce.

The maths lives in `scripts/check_espirit_acs.py` and is imported, not copied;
this notebook is the interactive front end to it. For a headless run over a
whole volume use the script directly:

    python -m scripts.check_espirit_acs --anatomy brain --split val --acs 20,48,96

**What the numbers mean**

| column | what it says | bad looks like |
|---|---|---|
| `rows/cols` | shape of the ESPIRiT calibration matrix | `< 1`: no null space is actually estimated |
| `support` / `brain unc` | where the maps are nonzero, and how much brain they miss | any brain uncovered is unrecoverable by any net |
| `\|RSS-1\|` | `operators/noise.py::mri_awgn` assumes `sum_c \|s_c\|^2 = 1` | not ~0 |
| `residual` | `\|\| c - s x \|\| / \|\| c \|\|`, the part of the coil data the SENSE model cannot represent | a floor no net can beat |
| `retention` | `\|\|x\|\| / \|\|RSS\|\|` | `< 1`: oversmoothed maps combining the coils incoherently |
| `phase coh` | smoothness of `arg x` | low at EVERY acs means the coil-0 phase reference is the problem, not the ACS |

Maps are defined only up to a per-pixel phase, so nothing here diffs maps
against maps -- that would measure the convention. Everything is scored through
the images the maps produce.

In [ ]:
%matplotlib inline
import os
import pathlib
import sys
import time

import numpy as np
import torch
import matplotlib.pyplot as plt

ROOT = pathlib.Path.cwd()
if not (ROOT / "physics").is_dir():
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

from operators.fourier import ifftc
from physics.smaps import espirit, espirit_soft
from scripts.check_espirit_acs import (
    KSPACE_ROOTS, calib_shape, cg_sense_nrmse, compare, crop_readout, draw,
    load_slices, map_metrics, pick_volume, subspace_residual,
)

print("root", ROOT)

In [ ]:
# ===================== WHAT TO COMPARE -- TUNE =====================
ANATOMY  = "brain"
SPLIT    = "val"
VOLUME   = None            # None -> first matching volume under KSPACE_ROOTS
SLICE    = "mid"           # "mid", "all", or e.g. "4,8,12"

# FIRST entry is treated as the shipped setting, LAST as the reference.
ACS_LIST = [(20, 20), (48, 48), (96, 96)]

KERNEL_SIZE     = 8        # the defaults are make_espirit_smaps.py's
THRESH_EIG      = 0.95
THRESH_ROWSPACE = 0.05
MAXIT           = 100      # espirit power-method iterations

CROP_READOUT = False       # drop brain's 2x readout oversampling first
MASK_THRESH  = 0.05        # brain mask is RSS > this, with RSS scaled to max 1
PHASE_KS     = 5           # phase-coherence boxcar side
DIFF_SCALE   = 0.2         # difference panels use +- this x the RSS window

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# ===================================================================
print(DEVICE)

## Load one slice

The slice is scaled so that `max(RSS) = 1`. Every metric below is a ratio and
does not care, but it makes the display windows and the CG-SENSE `lamda`
comparable between volumes.

In [ ]:
path = VOLUME or pick_volume(KSPACE_ROOTS[ANATOMY].format(split=SPLIT), ANATOMY, 0)
kvol, sl_idx = load_slices(path, SLICE)
if CROP_READOUT:
    kvol = crop_readout(kvol)

SI = 0                                    # which of the loaded slices to work on
C, Nx, Ny = kvol.shape[1], kvol.shape[-2], kvol.shape[-1]

k = kvol[SI:SI + 1].to(DEVICE)
coil = ifftc(k)
rss = coil.abs().pow(2).sum(1, keepdim=True).sqrt()
scale = rss.amax().clamp_min(1e-12)
k, coil, rss = k / scale, coil / scale, rss / scale
brain = rss[:, 0] > MASK_THRESH

print(f"{os.path.basename(path)}   slice {sl_idx[SI]} of {sl_idx}")
print(f"  grid {(Nx, Ny)}   coils {C}   "
      f"brain {float(brain.float().mean()):.1%} of FOV"
      f"{'   (readout oversampling cropped)' if CROP_READOUT else ''}")

## 1. The calibration geometry, before computing anything

ESPIRiT splits the calibration matrix's row space from its null space. The
matrix has `(ax-ks+1)(ay-ks+1)` rows -- one per ACS patch -- and `ks^2 * C`
columns. When there are fewer rows than columns there is no estimated null
space at all: the SVD returns at most `rows` singular vectors spanning a
`cols`-dimensional kernel space, and the retained subspace is whatever those
few patches happened to span.

This cell computes nothing and is the fastest way to find out that a setting
was never going to work.

In [ ]:
cols = KERNEL_SIZE ** 2 * C
print(f"calibration matrix   cols = ks^2 x C = {cols}")
print(f"  {'acs':>9}{'rows':>8}{'rows/cols':>11}{'map res (px)':>15}   verdict")
for acs in ACS_LIST:
    rows, _ = calib_shape(acs, KERNEL_SIZE, C)
    res = f"{Nx / max(acs[0], 1):.0f} x {Ny / max(acs[1], 1):.0f}"
    verdict = ("IMPOSSIBLE -- the kernel does not fit in the ACS" if rows == 0
               else "UNDERDETERMINED -- no null space is actually estimated"
               if rows < cols else "ok" if rows >= 2 * cols else "marginal")
    print(f"  {str(acs[0]) + 'x' + str(acs[1]):>9}{rows:>8}{rows / cols:>11.2f}"
          f"{res:>15}   {verdict}")

need = KERNEL_SIZE - 1 + int(np.ceil(np.sqrt(cols)))
print(f"  square ACS needs >= {need} per side for rows >= cols at "
      f"ks={KERNEL_SIZE}, C={C}")

## 2. Estimate and score

`runs` keeps each map set and its coil-combined image, so later cells (and you)
can poke at them.

**Memory.** The kernel images are `coils x retained kernels x the full grid`,
complex, and the retained-kernel count grows with the ACS -- gigabytes per
slice on brain's 640x320 grid, which is why the generator chunks. An OOM here
is caught and reported per setting rather than losing the whole run; if one
fails, set `CROP_READOUT = True` or drop the largest ACS and re-run.

In [ ]:
runs = []
print(f"  {'acs':>9}{'support':>9}{'brain unc':>11}{'|RSS-1|':>10}{'residual':>10}"
      f"{'retention':>11}{'p5 |x|/RSS':>12}{'phase coh':>11}{'sec':>7}")

for acs in ACS_LIST:
    tag = f"{acs[0]}x{acs[1]}"
    rows, _ = calib_shape(acs, KERNEL_SIZE, C)
    if rows == 0:
        print(f"  {tag:>9}  skipped: no patch fits -- kernel_size "
              f"{KERNEL_SIZE} exceeds the ACS")
        runs.append(None)
        continue

    ub = C * min(rows, cols) * Nx * Ny * 8 / 2 ** 30
    t0 = time.time()
    try:
        sm = espirit(k, acs_size=acs, kernel_size=KERNEL_SIZE,
                     thresh_rowspace=THRESH_ROWSPACE, thresh_eig=THRESH_EIG,
                     maxit=MAXIT)
    except (torch.cuda.OutOfMemoryError, RuntimeError, MemoryError) as e:
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()
        print(f"  {tag:>9}  FAILED: {type(e).__name__}: "
              f"{str(e).splitlines()[0][:80]}")
        print(f"            kernel images need up to {ub:.1f} GB here")
        runs.append(None)
        continue

    r = map_metrics(sm, coil, rss, brain, PHASE_KS)
    r["sm"] = sm
    runs.append(r)
    print(f"  {tag:>9}{r['support']:>9.1%}{r['uncovered']:>11.2%}"
          f"{r['rss_err']:>10.1e}{r['residual']:>10.4f}{r['retention']:>11.4f}"
          f"{r['p5']:>12.3f}{r['phase_coh']:>11.3f}{time.time() - t0:>7.1f}")

live = [(a, r) for a, r in zip(ACS_LIST, runs) if r is not None]

## 3. The actual question: do the coil-combined images differ?

Two numbers, because they answer different things. The **magnitude** NRMSE is
what a magnitude-domain metric (PSNR/SSIM on `|x|`) would see. The **complex**
one removes a single global phase first -- the whole image rotating is a
convention difference and harmless; anything left is a per-pixel phase
disagreement, which a complex-valued net sees as structure.

In [ ]:
if len(live) >= 2:
    ref_acs, ref = live[-1]
    print(f"vs the acs {ref_acs[0]}x{ref_acs[1]} reference, over the brain:")
    print(f"  {'acs':>9}{'|x| NRMSE':>12}{'complex NRMSE':>16}")
    for acs, r in live[:-1]:
        mag, cpx = compare(r["x"], ref["x"], brain)
        print(f"  {str(acs[0]) + 'x' + str(acs[1]):>9}{mag:>12.4f}{cpx:>16.4f}")
else:
    print("need at least two surviving settings to compare")

## 4. Look at them

Every magnitude panel shares ONE window taken from the RSS of this slice, and
every difference panel shares one symmetric window at `DIFF_SCALE` times that.
Nothing is autoscaled per panel: a map set that loses 30% of the signal has to
LOOK 30% darker instead of being renormalised back into agreement. A blank
difference panel is therefore ambiguous on its own, so each one reports its
peak in the title.

Rows: coil-combined magnitude, support, `|s|` for coil 0, `arg x`, and the
difference against the RSS. The bottom-left panel is the shipped ACS minus the
reference ACS -- literally the question this notebook asks.

In [ ]:
fig = draw(rss, brain, runs, ACS_LIST, coil,
           f"{os.path.basename(path)}  slice {sl_idx[SI]}   ks={KERNEL_SIZE} "
           f"eig={THRESH_EIG} rowspace={THRESH_ROWSPACE}",
           DIFF_SCALE)

## 5. The maps used the way training uses them

A CG-SENSE reconstruction at `R`, scored against the fully sampled RSS inside
the brain. This is the end-to-end consequence of everything above: an unrolled
net handed these maps inherits whatever the forward model gets wrong.

`lamda` is not optional -- at `lamda = 0` the normal operator is singular
wherever the maps vanish and CG wanders in that null space.

In [ ]:
RECON_R         = 4
RECON_ACS_LINES = 24
RECON_LAMDA     = 1.0e-3
RECON_ITERS     = 64

print(f"CG-SENSE R={RECON_R} (acs_lines={RECON_ACS_LINES}, "
      f"lamda={RECON_LAMDA:g}), NRMSE vs fully sampled RSS:")
for acs, r in live:
    try:
        e = cg_sense_nrmse(r["sm"], k, rss, brain, RECON_R, RECON_ACS_LINES,
                           RECON_LAMDA, RECON_ITERS)
        print(f"  {str(acs[0]) + 'x' + str(acs[1]):>9}{e:>12.4f}")
    except Exception as exc:
        print(f"  {str(acs[0]) + 'x' + str(acs[1]):>9}  skipped: {exc}")

## 6. Is it the calibration, or the model?

A single map per coil says the coil data lies in a rank-1 subspace at every
pixel. Where that is false -- aliasing from outside the FOV, motion, fat/water
chemical shift -- no amount of calibration fixes it, and a second set of maps
does. `espirit_soft` returns `M` maps and this scores the same residual against
their span.

If the 1-map residual is high and the 2-map one is not, the model is the
problem and a bigger ACS will not help. That is also one of the things an
E2E-VarNet's learned maps can absorb.

In [ ]:
SOFT_MAPS = 2

print(f"  {'acs':>9}{'1-map':>10}{str(SOFT_MAPS) + '-map':>10}")
for acs, r in live:
    try:
        sms = espirit_soft(k, acs_size=acs, kernel_size=KERNEL_SIZE,
                           thresh_rowspace=THRESH_ROWSPACE,
                           thresh_eig=THRESH_EIG, num_maps=SOFT_MAPS)
        rs = subspace_residual(sms, coil, brain)
        del sms
        print(f"  {str(acs[0]) + 'x' + str(acs[1]):>9}{r['residual']:>10.4f}{rs:>10.4f}")
    except (torch.cuda.OutOfMemoryError, RuntimeError, MemoryError) as e:
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()
        print(f"  {str(acs[0]) + 'x' + str(acs[1]):>9}  skipped: {type(e).__name__}")

## 7. Sweep

`THRESH_EIG` sets the support and the ACS sets how much calibration data the
kernel sees. These two move the residual most. This is the slow cell -- one
full ESPIRiT per row.

In [ ]:
SWEEP_ACS = [(20, 20), (32, 32), (48, 48), (64, 64)]
SWEEP_EIG = [0.90, 0.95, 0.98]

print(f"  {'acs':>9}{'eig':>7}{'support':>10}{'brain unc':>11}"
      f"{'residual':>10}{'retention':>11}")
for acs in SWEEP_ACS:
    for eig in SWEEP_EIG:
        try:
            S = espirit(k, acs_size=acs, kernel_size=KERNEL_SIZE,
                        thresh_rowspace=THRESH_ROWSPACE, thresh_eig=eig,
                        maxit=MAXIT)
        except (torch.cuda.OutOfMemoryError, RuntimeError, MemoryError) as e:
            if DEVICE.type == "cuda":
                torch.cuda.empty_cache()
            print(f"  {str(acs[0]) + 'x' + str(acs[1]):>9}{eig:>7.2f}"
                  f"   skipped: {type(e).__name__}")
            continue
        m = map_metrics(S, coil, rss, brain, PHASE_KS)
        print(f"  {str(acs[0]) + 'x' + str(acs[1]):>9}{eig:>7.2f}"
              f"{m['support']:>10.1%}{m['uncovered']:>11.2%}"
              f"{m['residual']:>10.4f}{m['retention']:>11.4f}")
        del S, m

## How to read it

**The residual and retention barely move between 20x20 and the large ACS.**
The ACS is not what is hurting the nets. Look at the phase-coherence column and
at section 6 -- if the 2-map residual is much lower, the single-map model is the
ceiling; if phase coherence is low everywhere, the coil-0 phase reference in
`physics/smaps.py::espirit` is putting noise into the ground-truth phase wherever
coil 0 is dark.

**The residual drops / retention rises with the larger ACS.** The shipped maps
are calibration-starved. Regenerate with a larger ACS into a NEW directory:

    python scripts/make_espirit_smaps.py --anatomy brain --split train --acs 48

and point `smap_root` at it. That script rewrites `image` as well as `smaps`,
which is the point -- the coil-combined ground truth is a function of the maps,
and keeping the old one would leave a file whose ground truth is not what its
own operator produces.

Either way, re-run this on a few volumes (`VOLUME = "..."`) before committing to
a regeneration: coil counts and anatomy placement vary across fastMRI brain, and
the calibration table's verdict depends on `C`.

## 8. The shipped ESPIRiT maps against fully sampled Walsh

Brain used to train on Walsh maps and did better than it does on ESPIRiT. This
section finds out whether that was **better maps** or **an easier / leakier
problem**. Five map sets on the same slice:

| tag | maps | what it stands for |
|---|---|---|
| `espirit-disk` | read from `SMAP_ROOTS["espirit"]` | **exactly what training uses now** |
| `espirit-20`   | recomputed here at the shipped ACS | checks the disk file is what we think |
| `espirit-<acs>` | the largest-ACS run from section 2 | best-case ESPIRiT on this slice |
| `walsh-full`   | `physics.smaps.walsh` on the FULLY sampled coil images | what brain used to be |
| `walsh-acs`    | the same, from the ACS lines only | Walsh with only the information an undersampled scan has |
| `walsh-disk`   | read from `SMAP_ROOTS["walsh"]`, if present | the maps the old brain runs actually trained on |

Three things could make Walsh look better:

1. **Leakage.** Walsh from fully sampled data has maps at the image's own
   resolution (5x5 patches), so the forward model carries detail the scan did
   not measure. `walsh-full` vs `walsh-acs` isolates this.
2. **Phase reference.** `espirit()` references every map to **coil 0**
   (`physics/smaps.py`, `ref = smaps[:, :1]`); `walsh()` references to the
   **strongest coil**. Wherever coil 0 is dark, ESPIRiT writes phase noise into
   the ground truth `x = S^H c`. A complex-valued unrolled net has to reproduce
   that phase; E2E-VarNet returns RSS magnitude and never sees it. That
   asymmetry would penalise exactly your nets and not VarNet.
3. **Support.** ESPIRiT zeroes pixels below `thresh_eig`; Walsh zeroes none. A
   hole in the brain is a hole in the ground truth.

In [ ]:
# ===================== SECTION 8 -- TUNE =====================
PRE = "/home/ee2178/scratch/ee2178/datasets/fastmri_preprocessed"
SMAP_ROOTS = {
    # make_espirit_smaps.py writes <split>_espirit; the old brain maps sit at <split>
    "espirit": PRE + "/brain_T2W_coil_combined/{split}_espirit",
    "walsh":   PRE + "/brain_T2W_coil_combined/{split}",
}
WALSH_KS, WALSH_STRIDE = 5, 2     # physics.smaps.walsh defaults
ACS_LINES   = 20                  # the recon configs' mri.acs_lines
ACS_WINDOW  = True                # Hann taper on the ACS lines before walsh-acs
# =============================================================
import h5py
from physics.smaps import walsh

fname = os.path.basename(path)
sl = sl_idx[SI]


def load_disk(kind):
    """(smaps (1,C,H,W), image (1,1,H,W) or None) for this slice, or None."""
    p = os.path.join(SMAP_ROOTS[kind].format(split=SPLIT), fname)
    if CROP_READOUT:
        print(f"  {kind}-disk: skipped -- CROP_READOUT changes the grid the "
              f"file was written on")
        return None
    if not os.path.exists(p):
        print(f"  {kind}-disk: no file at {p}")
        return None
    with h5py.File(p, "r") as f:
        keys = list(f.keys())
        if "smaps" not in f:
            print(f"  {kind}-disk: no `smaps` in {p} (keys {keys})")
            return None
        sm = torch.from_numpy(np.asarray(f["smaps"][sl])).to(torch.complex64)
        im = (torch.from_numpy(np.asarray(f["image"][sl])).to(torch.complex64)
              if "image" in f else None)
        attrs = dict(f.attrs)
    if tuple(sm.shape) != tuple(coil.shape[1:]):
        print(f"  {kind}-disk: map shape {tuple(sm.shape)} does not match the "
              f"coil images {tuple(coil.shape[1:])} -- skipped")
        return None
    if attrs:
        print(f"  {kind}-disk attrs: " + ", ".join(
            f"{a}={attrs[a]}" for a in attrs if a != "source_kspace"))
    im = None if im is None else im.reshape(1, 1, *im.shape[-2:]).to(DEVICE) / scale
    return sm[None].to(DEVICE), im


# walsh-acs: only the ACS phase-encode lines (full readout), which is exactly
# what the undersampled scan contains at the centre of k-space
W_ = k.shape[-1]
lo = W_ // 2 - ACS_LINES // 2
acs_mask = torch.zeros(W_, device=DEVICE)
acs_mask[lo:lo + ACS_LINES] = (torch.hann_window(ACS_LINES + 2, periodic=False,
                                                 device=DEVICE)[1:-1]
                               if ACS_WINDOW else 1.0)
coil_acs = ifftc(k * acs_mask)

maps = {}
d = load_disk("espirit")
if d is not None:
    maps["espirit-disk"] = d
maps["espirit-20"] = (live[0][1]["sm"] if live and live[0][0] == (20, 20)
                      else espirit(k, acs_size=(20, 20), kernel_size=KERNEL_SIZE,
                                   thresh_rowspace=THRESH_ROWSPACE,
                                   thresh_eig=THRESH_EIG, maxit=MAXIT), None)
if live and live[-1][0] != (20, 20):
    maps[f"espirit-{live[-1][0][0]}"] = (live[-1][1]["sm"], None)
maps["walsh-full"] = (walsh(coil, ks=WALSH_KS, stride=WALSH_STRIDE), None)
maps["walsh-acs"] = (walsh(coil_acs, ks=WALSH_KS, stride=WALSH_STRIDE), None)
d = load_disk("walsh")
if d is not None:
    maps["walsh-disk"] = d
print("map sets:", list(maps))

### 8a. Is the disk file what we think it is?

The ground truth on disk must equal `S_disk^H c` computed here, and the disk maps
should combine the coils the same way a fresh ESPIRiT at the shipped settings
does. Either failing means the training data is not what every other number in
this notebook assumes -- stop and look at that first.

In [ ]:
def nrmse(a, b, m):
    a, b = a[:, 0][m], b[:, 0][m]
    return float((a - b).norm() / b.norm().clamp_min(1e-12))

for kind in ("espirit-disk", "walsh-disk"):
    if kind not in maps:
        continue
    sm, im = maps[kind]
    x_here = (sm.conj() * coil).sum(1, keepdim=True)
    if im is not None:
        # a pure scalar mismatch (e.g. a preprocessing scale) is harmless, so
        # report it separately from the shape of the disagreement
        a_, b_ = im[:, 0][brain], x_here[:, 0][brain]
        gain = complex((b_.conj() * a_).sum() / (b_.abs().pow(2).sum() + 1e-12))
        print(f"{kind}: || image_disk - S^H c || / || S^H c ||  (brain) = "
              f"{nrmse(im, x_here, brain):.2e}   best-fit gain {abs(gain):.4f} "
              f"(after it: {nrmse(im / gain, x_here, brain):.2e})")
    if kind == "espirit-disk":
        mag, cpx = compare(x_here, (maps['espirit-20'][0].conj() * coil).sum(1, keepdim=True), brain)
        print(f"{kind} vs espirit-20 recomputed: |x| NRMSE {mag:.2e}   complex {cpx:.2e}")

### 8b. The maps against the fully sampled data

Same columns as section 2. `vs walsh-full` compares each coil-combined image to
the fully sampled Walsh combination, magnitude and complex (one global phase
removed).

In [ ]:
res8 = {}
xw = (maps["walsh-full"][0].conj() * coil).sum(1, keepdim=True)
print(f"  {'maps':>14}{'support':>9}{'brain unc':>11}{'|RSS-1|':>10}{'residual':>10}"
      f"{'retention':>11}{'p5 |x|/RSS':>12}{'phase coh':>11}{'|x| vs W':>10}{'cpx vs W':>10}")
for tag, (sm, _) in maps.items():
    r = map_metrics(sm, coil, rss, brain, PHASE_KS)
    mag, cpx = compare(r["x"], xw, brain)
    res8[tag] = r
    print(f"  {tag:>14}{r['support']:>9.1%}{r['uncovered']:>11.2%}"
          f"{r['rss_err']:>10.1e}{r['residual']:>10.4f}{r['retention']:>11.4f}"
          f"{r['p5']:>12.3f}{r['phase_coh']:>11.3f}{mag:>10.4f}{cpx:>10.4f}")

### 8c. The problem each map set poses at the training operating point

Two CG-SENSE runs per map set, at the grid's R and ACS:

- **measured**: the real undersampled k-space. How well the maps model the data
  the scanner produced. Scored against the fully sampled RSS.
- **simulated**: what training actually does (`kspace_type: "simulated"`):
  `x_gt = S^H c`, `y = M F S x_gt (+ noise)`. Scored against that map set's OWN
  ground truth. This is the difficulty of the task the nets were trained on -- if
  Walsh makes it much easier, "Walsh did better" was a statement about the task,
  not about the nets.

`NOISE` is relative to this slice's max RSS = 1, NOT the training `noise_std`
scale (that one is after the loader's `scale_fac`), so read the columns against
each other, not against training numbers.

In [ ]:
from physics.gfactor import cg_sense
from physics.mask import make_acc_mask
from operators.fourier import fftc

RECON_R8    = 16
NOISE       = [0.0, 0.02]
LAMDA8      = 1.0e-3
ITERS8      = 64
g = torch.Generator(device=DEVICE).manual_seed(0)

H_, W_ = k.shape[-2:]
m8 = make_acc_mask((H_, W_), accel=RECON_R8, acs_lines=ACS_LINES, dim=1,
                   mode="uniform", device=DEVICE)

hdr = "".join(f"{'sim s=' + str(s):>12}" for s in NOISE)
print(f"CG-SENSE R={RECON_R8}, acs_lines={ACS_LINES}, NRMSE of |x| over the brain")
print(f"  {'maps':>14}{'measured':>12}{hdr}")
for tag, (sm, _) in maps.items():
    rec = cg_sense(sm, m8, lamda=LAMDA8, max_iter=ITERS8)
    xm = rec(m8 * k)
    row = f"  {tag:>14}{nrmse(xm.abs(), rss, brain):>12.4f}"
    xgt = (sm.conj() * coil).sum(1, keepdim=True)
    for s in NOISE:
        cim = sm * xgt
        if s > 0:
            n = torch.complex(torch.randn(cim.shape, generator=g, device=DEVICE),
                              torch.randn(cim.shape, generator=g, device=DEVICE))
            cim = cim + s / 2 ** 0.5 * n
        xs = rec(m8 * fftc(cim))
        row += f"{nrmse(xs.abs(), xgt.abs(), brain):>12.4f}"
    print(row)
    del rec, xm

In [ ]:
# rows: map sets. cols: |x|, |x| - RSS, arg x, |s| of coil 0, support.
# One magnitude window from the RSS, one symmetric difference window -- nothing
# autoscaled, so lost signal LOOKS lost.
tags = list(maps)
V = float(rss.amax())
fig, ax = plt.subplots(len(tags), 5, figsize=(15, 3.1 * len(tags)), squeeze=False)
for i, tag in enumerate(tags):
    sm = maps[tag][0]
    x = res8[tag]["x"][0, 0]
    dif = (x.abs() - rss[0, 0]) * brain[0]
    panels = [
        (x.abs(), dict(cmap="gray", vmin=0, vmax=V), "|x|"),
        (dif, dict(cmap="RdBu_r", vmin=-DIFF_SCALE * V, vmax=DIFF_SCALE * V),
         f"|x|-RSS  peak {float(dif.abs().max()):.3f}"),
        (x.angle() * (x.abs() > 1e-6 * V), dict(cmap="twilight", vmin=-np.pi, vmax=np.pi),
         f"arg x  coh {res8[tag]['phase_coh']:.3f}"),
        (sm[0, 0].abs(), dict(cmap="gray", vmin=0, vmax=1), "|s| coil 0"),
        ((sm.abs().pow(2).sum(1)[0] > 1e-12).float(), dict(cmap="gray", vmin=0, vmax=1),
         f"support {res8[tag]['support']:.0%}"),
    ]
    for j, (img, kw, t) in enumerate(panels):
        ax[i][j].imshow(img.detach().cpu(), **kw)
        ax[i][j].set_title(f"{tag}: {t}" if j == 0 else t, fontsize=9)
        ax[i][j].axis("off")
plt.suptitle(f"{fname}  slice {sl}", y=1.0)
plt.tight_layout()
plt.show()

### How to read section 8

- **8a fails** (image_disk differs from `S^H c`, or disk ESPIRiT differs from
  `espirit-20`): the training files are not what they should be. Fix that before
  anything else -- e.g. a map/image pair from different runs, or the kernel-flip
  fix applied to one and not the other.
- **`walsh-full` beats `walsh-acs` by a lot** (residual, measured CG-SENSE):
  the Walsh advantage is information from the fully sampled data leaking into
  the forward model. That is not a legitimate setting for an accelerated-recon
  benchmark, and the old brain numbers were easier than the current ones.
- **Simulated CG-SENSE is much lower with Walsh:** the old training task was
  easier, so the drop after switching is expected and says nothing about the
  nets. Numbers from the two map sets must not share a table.
- **ESPIRiT's phase coherence is well below Walsh's**, with dark-coil-0 regions
  visible in `arg x`: the coil-0 phase reference is putting noise into the
  ground-truth phase. That penalises the complex unrolled nets and not VarNet
  (magnitude output). The fix is a better reference in `espirit()` (strongest
  coil, or a smooth phase from the low-res combined image) and regenerating.
- **ESPIRiT shows `brain unc` > 0 or retention well below 1:** the eigenvalue
  threshold is cutting the object; try `THRESH_EIG` lower in section 7.
- **`walsh-acs` is as good as `walsh-full` and beats ESPIRiT:** Walsh is simply
  better here, legitimately -- consider generating Walsh-from-ACS maps (they
  have no hard zeros, so the organ mask needs another definition).

## 9. Does the map support agree with the object?

The mask every masked metric is computed over used to be the coil-map support.
It is now the RSS object mask (`physics/object_mask.py`), because the support is
dilated by construction. This section MEASURES that disagreement instead of
asserting it, per map set:

| column | meaning |
|---|---|
| `dice`, `iou` | overlap of the two masks; 1.0 = identical |
| `sup only` | in the support, NOT in the object -- the dilation, as a fraction of the object's area. **This is the air that masked metrics were scoring.** |
| `obj only` | in the object, NOT in the support -- holes the eigenvalue threshold punched in the anatomy. Those pixels are unrecoverable: `image` is identically zero there. |
| `dilation px` | erosions of the support needed before it fits inside the object (95% and 100% of the excess gone). Read it as the boundary offset in pixels. |

The RSS mask is the reference here, not because it is beyond question, but
because it is what the object's own edge says and it uses no maps at all.

In [ ]:
from physics.object_mask import rss_object_mask, erode

# The mask the loader now builds, from this slice's RSS. `rss` is already
# scaled to max 1, which changes nothing: the threshold is Otsu's, per sample.
obj_mask = rss_object_mask(rss)                      # (1, H, W) bool


def dilation_px(sup, obj, max_r=64):
    """Erosions of `sup` until its excess over `obj` is 95% / 100% gone."""
    excess0 = int((sup & ~obj).sum())
    if excess0 == 0:
        return 0, 0
    r95 = r100 = None
    for r in range(1, max_r + 1):
        e = erode(sup, r)
        excess = int((e & ~obj).sum())
        if r95 is None and excess <= 0.05 * excess0:
            r95 = r
        if excess == 0:
            r100 = r
            break
    return r95, (r100 if r100 is not None else float("inf"))


print(f"RSS object mask keeps {float(obj_mask.float().mean()):>6.1%} of the FOV")
print(f"  {'maps':>14}{'support':>9}{'dice':>8}{'iou':>8}"
      f"{'sup only':>10}{'obj only':>10}{'dilation px':>14}")
agree9 = {}
for tag, (sm, _) in maps.items():
    sup = (sm.abs().pow(2).sum(1) > 0)               # (1, H, W)
    inter = float((sup & obj_mask).sum())
    dice = 2 * inter / max(float(sup.sum()) + float(obj_mask.sum()), 1)
    iou = inter / max(float((sup | obj_mask).sum()), 1)
    sup_only = float((sup & ~obj_mask).sum()) / max(float(obj_mask.sum()), 1)
    obj_only = float((obj_mask & ~sup).sum()) / max(float(obj_mask.sum()), 1)
    r95, r100 = dilation_px(sup, obj_mask)
    agree9[tag] = dict(sup=sup, dice=dice, iou=iou, sup_only=sup_only,
                       obj_only=obj_only, r95=r95, r100=r100)
    print(f"  {tag:>14}{float(sup.float().mean()):>9.1%}{dice:>8.3f}{iou:>8.3f}"
          f"{sup_only:>10.1%}{obj_only:>10.2%}{f'{r95} / {r100}':>14}")

In [ ]:
# Boundaries, so the disagreement is visible rather than only tabulated:
# RSS object mask in green, map support in red, over the RSS.
def outline(m):
    m4 = m[None].float() if m.dim() == 3 else m.float()
    import torch.nn.functional as _F
    er = _F.max_pool2d(-_F.pad(m4, (1,) * 4, value=0.0), 3, stride=1) * -1
    return (m4 - er)[0, 0].cpu() > 0

tags9 = list(agree9)
V = float(rss.amax())
fig, ax = plt.subplots(1, len(tags9), figsize=(4.2 * len(tags9), 4.6), squeeze=False)
base = (rss[0, 0] / V).clamp(0, 1).cpu()
og = outline(obj_mask)
for j, tag in enumerate(tags9):
    rgb = base[..., None].repeat(1, 1, 3)
    rgb[outline(agree9[tag]["sup"])] = torch.tensor([1.0, 0.15, 0.15])
    rgb[og] = torch.tensor([0.15, 1.0, 0.3])
    ax[0][j].imshow(rgb.numpy())
    ax[0][j].set_title(f"{tag}\ndice {agree9[tag]['dice']:.3f}   "
                       f"dilation {agree9[tag]['r95']} px", fontsize=9)
    ax[0][j].axis("off")
plt.suptitle(f"green = RSS object mask, red = map support   {fname} slice {sl}", y=1.02)
plt.tight_layout()
plt.show()

### How to read section 9

- **`sup only` is large (tens of %) and `dilation px` is more than a few:** the
  support is the dilated boundary this was looking for, and every masked metric
  computed under `organ_mask_source="smaps"` was averaging over that much air.
  The RSS mask is the fix; the old masked numbers are not comparable to new ones.
- **`dice` is near 1 and `dilation px` is 0-2:** the support is fine on this
  slice and the mask change is cosmetic. Check a few more slices and volumes
  (`SLICE = "all"`, other `VOLUME`s) before concluding that, since the dilation
  scales with `N / kernel_size` and with how much of the FOV the object fills.
- **`obj only` is nonzero:** the eigenvalue threshold is cutting anatomy. Those
  pixels have `image == 0`, so no net can recover them and every model is
  charged for them equally. Lower `THRESH_EIG` (section 7) or use
  `organ_mask_source="rss+smaps"`, which excludes them from the metric.
- **`walsh-*` rows show `support` = 100%:** expected, not a result. Walsh writes
  no exact zeros, so its "support" is the whole FOV and its dice is just the
  object's area fraction. The row is there as the null case.